# Food Wastage Analysis Project
**Goal:** Understand and predict food wastage percentage in a college/hostel setting using daily meal data.

**Dataset:** Daily records of meals served, food cooked, food wasted, attendance, and other factors.

**Target Variable:** `Wastage %` (how much of the cooked food was wasted, in percentage)

---

## Phase 1 — Data Loading & Exploration

In [ ]:
# Importing all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("All libraries loaded successfully!")

In [ ]:
# Loading the dataset
raw_data = pd.read_csv('final_data.csv')
df = raw_data.copy()  # Always work on a copy, keep original safe

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Checking column types and basic info
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe().round(2)

In [ ]:
# Checking for missing values
print("Missing values in each column:")
print(df.isnull().sum())

> **Observation:** The dataset has 360 rows and 10 columns. There are no missing values, which means no imputation is needed. The target variable `Wastage %` ranges from ~8.8% to ~24%, with a mean of ~16.3%.

---
## Phase 2 — Data Preprocessing

In [ ]:
# Step 1: Fix the Date column — remove the extra ' tt' text and convert to datetime
df['Date'] = df['Date'].str.replace(' tt', '', regex=False)
df['Date'] = pd.to_datetime(df['Date'])

print("Date column fixed. Sample:")
print(df['Date'].head(3))

> **Note:** The raw date values had a ' tt' suffix (e.g. '10/1/2025 0:00 tt'). We removed it and converted to proper datetime format.

In [ ]:
# Step 2: Handle outliers using IQR capping
# We cap extreme values instead of removing rows (keeps all 360 rows)

numeric_cols = ['Attendance', 'Food Cooked (kg)', 'Food Wasted (kg)', 'Wastage %', 'Food Rating']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_count = ((df[col] < lower) | (df[col] > upper)).sum()
    if outliers_count > 0:
        df[col] = df[col].clip(lower=lower, upper=upper)
        print(f"{col}: {outliers_count} outliers capped")
    else:
        print(f"{col}: No outliers found")

> **Note:** Outliers are capped (not removed) using the IQR method. This means extreme values are replaced with the boundary values instead of deleting rows.

In [ ]:
# Step 3: Encode categorical columns into numbers
# Machine learning models need numbers, not text

# Label Encoding for Meal (Breakfast=0, Dinner=1, Lunch=2)
le_meal = LabelEncoder()
df['Meal_encoded'] = le_meal.fit_transform(df['Meal'])

# Label Encoding for Academic Pressure (Holidays=0, Normal=1)
le_pressure = LabelEncoder()
df['Pressure_encoded'] = le_pressure.fit_transform(df['Academic Pressure'])

# Label Encoding for Day of week
le_day = LabelEncoder()
df['Day_encoded'] = le_day.fit_transform(df['Day'])

print("Encoding done!")
print("Meal classes:", list(le_meal.classes_))
print("Pressure classes:", list(le_pressure.classes_))
print("Day classes:", list(le_day.classes_))

> **Note:** We converted text categories into numbers using LabelEncoder. This is required for regression models to work.

---
## Phase 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Plot 1: Average Wastage % by Meal
meal_wastage = df.groupby('Meal')['Wastage %'].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
meal_wastage.plot(kind='bar', color=['#e74c3c', '#f39c12', '#3498db'], edgecolor='black')
plt.title('Average Wastage % by Meal Type')
plt.xlabel('Meal')
plt.ylabel('Average Wastage %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(meal_wastage)

> **Finding:** Dinner tends to have the highest food wastage. This could be because dinner portions are planned for more people but attendance drops in the evening.

In [ ]:
# Plot 2: Average Wastage % by Day of Week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_wastage = df.groupby('Day')['Wastage %'].mean().reindex(day_order)

plt.figure(figsize=(8, 4))
day_wastage.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Wastage % by Day of Week')
plt.xlabel('Day')
plt.ylabel('Average Wastage %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Finding:** Weekends (Saturday/Sunday) show higher wastage, likely because fewer students are present in the hostel.

In [ ]:
# Plot 3: Wastage % during Holidays vs Normal days
plt.figure(figsize=(6, 4))
sns.boxplot(x='Academic Pressure', y='Wastage %', data=df, palette='Set2')
plt.title('Wastage % : Holidays vs Normal Days')
plt.xlabel('Period Type')
plt.ylabel('Wastage %')
plt.tight_layout()
plt.show()

print(df.groupby('Academic Pressure')['Wastage %'].mean())

> **Finding:** Holiday periods show noticeably higher wastage than normal days. This matches expectation — food is still cooked in bulk but fewer students are present.

In [ ]:
# Plot 4: Correlation Heatmap
corr_cols = ['Attendance', 'Food Cooked (kg)', 'Wastage %', 'Food Rating', 'Menu Category']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

> **Finding:** Attendance is negatively correlated with Wastage % — when more students show up, less food is wasted. Food Rating is also negatively correlated — better-rated food gets eaten more.

In [ ]:
# Plot 5: Time-Series — Wastage % over Time
daily_wastage = df.groupby('Date')['Wastage %'].mean().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(daily_wastage['Date'], daily_wastage['Wastage %'], color='tomato', linewidth=1.5)
plt.axhline(y=daily_wastage['Wastage %'].mean(), color='gray', linestyle='--', label='Average')
plt.title('Daily Average Wastage % Over Time')
plt.xlabel('Date')
plt.ylabel('Wastage %')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Finding:** Wastage % fluctuates daily but stays mostly around the 16% average. There are some spikes which likely correspond to weekends or holiday periods.

In [ ]:
# Plot 6: Distribution of the Target Variable — Wastage %
plt.figure(figsize=(7, 4))
plt.hist(df['Wastage %'], bins=20, color='steelblue', edgecolor='black')
plt.title('Distribution of Wastage %')
plt.xlabel('Wastage %')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f"Mean: {df['Wastage %'].mean():.2f}%")
print(f"Median: {df['Wastage %'].median():.2f}%")
print(f"Std Dev: {df['Wastage %'].std():.2f}%")

> **Finding:** The target variable `Wastage %` is roughly normally distributed with a mean of ~16.3%. This is a good sign for regression models.

---
## Phase 4 — Model Building

**Target:** `Wastage %`

**Features used:** Attendance, Food Cooked (kg), Food Rating, Menu Category, Meal, Day, Academic Pressure

In [ ]:
# Defining Features (X) and Target (y)

features = [
    'Attendance',
    'Food Cooked (kg)',
    'Food Rating',
    'Menu Category',
    'Meal_encoded',
    'Day_encoded',
    'Pressure_encoded'
]

X = df[features]
y = df['Wastage %']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns used:", features)

In [ ]:
# Splitting data into Training set (80%) and Test set (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Scaling the features (makes all features on the same scale)
# This helps Linear Regression work better

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled = scaler.transform(X_test)         # apply same scale to test

print("Scaling done!")

> **Note:** We fit the scaler only on training data and then apply it to test data. This prevents data leakage.

### Model 1 — Linear Regression

In [ ]:
# Training Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predicting on test set
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluation
lr_r2    = r2_score(y_test, y_pred_lr)
lr_mae   = mean_absolute_error(y_test, y_pred_lr)
lr_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("=== Linear Regression Results ===")
print(f"R²   : {lr_r2:.4f}")
print(f"MAE  : {lr_mae:.4f} %")
print(f"RMSE : {lr_rmse:.4f} %")

> **Note:** R² tells how much of the variation in Wastage % the model can explain. MAE and RMSE tell us the average error in percentage points.

In [ ]:
# Actual vs Predicted — Linear Regression
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_lr, color='steelblue', alpha=0.6, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Wastage %')
plt.ylabel('Predicted Wastage %')
plt.title('Linear Regression: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

### Model 2 — Random Forest Regressor

In [ ]:
# Training Random Forest model
# Random Forest does NOT need scaling, so we use original (unscaled) data
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predicting on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluation
rf_r2   = r2_score(y_test, y_pred_rf)
rf_mae  = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("=== Random Forest Results ===")
print(f"R²   : {rf_r2:.4f}")
print(f"MAE  : {rf_mae:.4f} %")
print(f"RMSE : {rf_rmse:.4f} %")

> **Note:** Random Forest builds many decision trees and averages their predictions. It usually performs better than Linear Regression on real-world data.

In [ ]:
# Actual vs Predicted — Random Forest
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_rf, color='seagreen', alpha=0.6, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Wastage %')
plt.ylabel('Predicted Wastage %')
plt.title('Random Forest: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance from Random Forest
importances = rf_model.feature_importances_
feat_names  = features

feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(7, 4))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='coral', edgecolor='black')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Finding:** The most important features for predicting Wastage % are shown above. Features with higher importance have more influence on the prediction.

---
## Phase 5 — Model Comparison

In [ ]:
# Final Comparison Table of all models
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R² Score': [round(lr_r2, 4), round(rf_r2, 4)],
    'MAE (%)': [round(lr_mae, 4), round(rf_mae, 4)],
    'RMSE (%)': [round(lr_rmse, 4), round(rf_rmse, 4)]
})

print(comparison.to_string(index=False))

In [ ]:
# Bar chart comparing both models on R² Score
plt.figure(figsize=(6, 4))
plt.bar(comparison['Model'], comparison['R² Score'],
        color=['steelblue', 'seagreen'], edgecolor='black')
plt.title('Model Comparison — R² Score')
plt.ylabel('R² Score')
plt.ylim(0, 1)
for i, val in enumerate(comparison['R² Score']):
    plt.text(i, val + 0.01, str(val), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

> **Conclusion:** Random Forest outperforms Linear Regression in all metrics — higher R² and lower MAE/RMSE. This is expected because food wastage is influenced by multiple non-linear factors.

---
## Phase 6 — Conclusion & Recommendations

### Key Findings from this Project

1. **Average wastage** is around **16.3%** of total food cooked — this is significant and should be reduced.

2. **Dinner** has the highest wastage compared to Breakfast and Lunch.

3. **Weekends and Holidays** show higher wastage because food is cooked in fixed quantities but fewer students are present.

4. **Higher attendance** leads to lower wastage — more students = more food consumed.

5. **Better food ratings** are linked to lower wastage — students eat more when the food quality is good.

6. **Random Forest** was the best model with a higher R² score, showing it can better capture the patterns in this dataset.

### Recommendations

- Reduce the quantity of food cooked on weekends and during holidays.
- Monitor and improve food quality (food rating) as it directly reduces wastage.
- Use the prediction model to estimate wastage in advance and adjust food preparation quantities accordingly.

---
*Project by: [Your Name] | Food Wastage Analysis using Machine Learning*